# LAB 12 - Random Forest for Regression

In this lab we will be extending the previous lab about Decision trees and build a Regression model using Random Forest.

For simplicity, we will be using the same dataset as the previous lab (you can find it in ECLASS).

**IMPORTANT:** For this lab, if you haven't finished your code from last week's lab on Decision trees, you will have the option to use the sklearn implementation for a regression tree. However, this doesn't mean that you should skip the previous lab. This is just so that you don't get behind with the content and you don't spend all your time today working on the previous lab. 

In [63]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
import pandas as pd

As mentioned before, use the Boston Housing data and prepare your train/val/test split as usual.

In [64]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep="\s+")

X = data.iloc[:, :-1].values
y = data["MEDV"].values

X_train, X_aux, y_train, y_aux = train_test_split(X, y, train_size= 0.8)
X_test, X_val, y_test, y_val = train_test_split(X_aux, y_aux, train_size= 0.5)


## Exercise 1 -- Bootstrap

Also known as [bagging](https://en.wikipedia.org/wiki/Bootstrap_aggregating), this technique consists of making several samples with replacement of the original data, using each of the samples to train an estimator, and then aggregating the predictions using the average (this is also a type of model ensemble).

In [65]:
def bootstrap(X, num_bags=10):
    """
    Given a dataset and a number of bags,
    sample the dataset with replacement.
    
    This function does not return a copy
    of the datapoints, but a list of indices
    with compatible dimensionality
    
    Parameters
    ----------
    X : ndarray
        A dataset
    num_bags : int, default 10
        The number of bags to create
    
    Returns
    -------
    list of ndarray
        The list contains `num_bags` integer one-dimensional ndarrays.
        Each of these contains the indices corresponding to the 
        sampled datapoints in `X`
    
    Notes
    -----
    * The number of datapoints in each bach will
      match the number of datapoints in the given
      dataset.
    * The
    """
    rng = np.random.default_rng(0) # you can change the seed, or use 0 to replicate my results
    
    n_samples = X.shape[0]
    return [rng.integers(0, n_samples, size=n_samples) for i in range(num_bags)]

In [66]:
rng = np.random.default_rng(0)
X_small = rng.random(size=(100,2))
bags = bootstrap(X_small)
bags[0]

array([85, 63, 51, 26, 30,  4,  7,  1, 17, 81, 64, 91, 50, 60, 97, 72, 63,
       54, 55, 93, 27, 81, 67,  0, 39, 85, 55,  3, 76, 72, 84, 17,  8, 86,
        2, 54,  8, 29, 48, 42, 40,  2,  0, 12,  0, 67, 52, 64, 25, 61, 76,
       38, 46, 99, 80, 98, 37, 68, 95, 65, 84, 68, 70, 38, 87, 13, 57, 72,
       84, 52, 37, 31, 42, 48, 71, 88,  7, 93, 53, 35, 67, 57, 25, 32, 71,
       59, 50, 33, 76, 39, 32, 89, 26, 22, 71, 62,  4,  8, 37, 83],
      dtype=int64)

In [67]:
def regression_criterion(region: np.ndarray):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """
    
    if region.size > 0: mean = region.mean()
    else: return float("inf")
    
    return np.sum((region - mean)**2)

def split_region(region, feature_index, tau):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    
    left_partition = region[:, feature_index] < tau
    right_partition = region[:, feature_index] >= tau
    
    return left_partition, right_partition

def get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    n_samples, n_features = X.shape
    
    best_sse = float("inf")
    best_feature = 0
    best_tau = 0
    best_left_indices = np.array([], dtype=int)
    best_right_indices = np.array([], dtype=int)
    
    for feature_idx in range(n_features):
        thresholds = X[:, feature_idx]

        for tau in thresholds:
            left_indices, right_indices = split_region(X, feature_idx, tau)
        
            current_sse = regression_criterion(y[left_indices]) + regression_criterion(y[right_indices])
            
            if current_sse < best_sse:
                best_sse = current_sse
                best_feature = feature_idx
                best_tau = tau
                best_left_indices = left_indices
                best_right_indices = right_indices
                
    return {'feature_index': int(best_feature),
            'tau': float(best_tau),
            'left_region': best_left_indices,
            'right_region': best_right_indices}
    

def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    
    left_indices = node["left_region"]
    right_indices = node["right_region"]
    
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    if (len(y_left) < min_samples or current_depth >= max_depth or regression_criterion(y_left) == 0):
        node["left_region"] = {"value": y_left.mean() if len(y_left) > 0 else 0.0}
    else:
        node["left_region"] = get_split(X_left, y_left)
        recursive_growth(node["left_region"], min_samples, max_depth, current_depth + 1, X_left, y_left)
    
    if (len(y_right) < min_samples or current_depth >= max_depth or regression_criterion(y_right) == 0):
        node["right_region"] = {"value": y_right.mean() if len(y_right) > 0 else 0.0}
    else:
        node["right_region"] = get_split(X_right, y_right)
        recursive_growth(node["right_region"], min_samples, max_depth, current_depth + 1, X_right, y_right)
        

def predict_sample(node, sample):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint
    """
    if "value" in node:
        return node["value"]
    if sample[node["feature_index"]] < node["tau"]:
        return predict_sample(node["left_region"], sample)
    else:
        return predict_sample(node["right_region"], sample)
    
        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    return np.array([predict_sample(node, sample) for sample in X])
        
    

In [68]:
def fit_bagging(X_train, y_train, num_bags, min_samples, max_depth):
    list_idx = bootstrap(X_train, num_bags)
    nodes = []

    for i, idx in enumerate(list_idx):
        X_bag = X_train[idx]
        y_bag = y_train[idx]

        node_bag = get_split(X_bag, y_bag)
        recursive_growth(node_bag, min_samples, max_depth, 1, X_bag, y_bag)
        nodes.append(node_bag)

    return nodes

def predict_bagging(trees, X):
    return [predict(tree, X) for tree in trees]


## Exercise 2 -- Aggregation

The second part of bagging.

In [69]:
def aggregate_regression(preds):
    """
    Aggregate predictions by several estimators
    
    Parameters
    ----------
    preds : list of ndarray
        Predictions from multiple estimators.
        All ndarrays in this list should have the same
        dimensionality.
        
    Return
    ------
    ndarray
        The mean of the predictions
    """

    P = np.column_stack(preds)
    return np.mean(P, axis= 1)

# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [70]:
trees = fit_bagging(X_train, y_train, 100, 5, 30)
preds = predict_bagging(trees, X_train)
y_pred_train = aggregate_regression(preds)
root_mean_squared_error(y_train, y_pred_train)

1.4657073863129553

In [71]:
preds = predict_bagging(trees, X_val)
y_pred_val = aggregate_regression(preds)
root_mean_squared_error(y_val, y_pred_val)

3.07854491973864

In [72]:
root = get_split(X_train, y_train)
min_samples = 20
max_depth = 6
recursive_growth(root, min_samples, max_depth, 1, X_train, y_train)
train_mse = root_mean_squared_error(y_train, predict(root, X_train))
test_mse = root_mean_squared_error(y_test, predict(root, X_test))

print(f'Train MSE : {train_mse}')
print(f'Test MSE : {test_mse}')

Train MSE : 2.197332889100299
Test MSE : 3.9207484738726355


## Exercise 3 -- Random Forest for regression

Using the functions you implemented above, it is now time to put all of them together to train several decision trees and then ensemble them to output a single prediction. For the random forest, however, we need to select a subset of features at each split on the decision tree. 

For this part, you can use the sklearn implementation of Decision trees for regression as your estimator for each set of features and bags. See below an example of how to do this, and always remember to check the necessary documentation when using an external function.

Some parameters you will have to set are: 
* num_features: number of features per estimator
* min_samples: min number of samples per leaf node
* max_depth: maximum depth of the decision tree (each estimator)
* num_estimators: number of decision trees you will create using each bag and random set of features

In [73]:
num_features = 7
min_samples = 10
max_depth = 10
num_estimators = 5

In [74]:
# example of sklearn Decision tree
estimator = DecisionTreeRegressor(max_depth=max_depth)
estimator.fit(X, y)
estimator.predict(X)

array([24.        , 21.4625    , 34.8       , 33.2       , 37.        ,
       28.7       , 20.14074074, 27.1       , 16.5       , 18.9       ,
       15.        , 18.16666667, 21.7       , 20.14074074, 19.97      ,
       20.14074074, 22.66666667, 17.5       , 21.70909091, 18.16666667,
       13.        , 18.16666667, 15.95      , 14.34      , 17.65294118,
       13.5       , 17.65294118, 14.61666667, 19.97      , 21.        ,
       12.86666667, 14.5       , 12.86666667, 14.61666667, 13.        ,
       21.70909091, 21.70909091, 21.70909091, 21.70909091, 30.8       ,
       34.8       , 26.6       , 24.30625   , 24.30625   , 20.14074074,
       18.14      , 20.14074074, 16.6       , 14.4       , 19.4       ,
       20.14074074, 21.4625    , 24.30625   , 20.14074074, 18.9       ,
       35.4       , 24.15      , 31.6       , 23.34      , 20.14074074,
       18.14      , 16.        , 22.55      , 24.66666667, 32.85      ,
       24.15      , 20.14074074, 20.14074074, 18.14      , 20.14

In [75]:
## your code goes here:
def fit_random_forest(X_train, y_train, num_estimators, num_features, min_samples, max_depth):
    list_idx = bootstrap(X_train, num_estimators)
    nodes = []

    for i, idx in enumerate(list_idx):
        X_bag = X_train[idx]
        y_bag = y_train[idx]

        DecisionTree = DecisionTreeRegressor(max_depth= max_depth, min_samples_leaf= min_samples, max_features= num_features)
        DecisionTree.fit(X_bag, y_bag)
        nodes.append(DecisionTree)

    return nodes

def predict_random_forest(trees, X):
    return [tree.predict(X) for tree in trees]

In [76]:
def r2_score(y_true, y_pred):
    sse_pred = np.sum((y_true - y_pred)**2)
    sse_mu = np.sum((y_true - y_true.mean())**2)
    
    return 1 - (sse_pred/sse_mu)

In [77]:
trees = fit_random_forest(X_train, y_train, 100, 4, 1, 13)
preds = predict_random_forest(trees, X_train)
y_pred_train = aggregate_regression(preds)
root_mean_squared_error(y_train, y_pred_train)
r2_score(y_train, y_pred_train)

0.9781276879618325

In [78]:
preds = predict_random_forest(trees, X_val)
y_pred_val = aggregate_regression(preds)
root_mean_squared_error(y_val, y_pred_val)
r2_score(y_val, y_pred_val)

0.9020102672640021